# Citi Velocity curves and swaption cubes: EOD and intraday timeseries

Multi-currency swap-curve timeseries at **end-of-day** and at **1-minute intraday**,
plus a **swaption-cube** timeseries, all served from warmed local stores.

## Nothing here touches Excel

Every cell below reads a **CurveStore / SwaptionCubeStore partition on disk**. That
matters for more than speed: it works when Excel is closed, it is reproducible (the
same request returns the same curve), and it cannot disturb a signed-in add-in.

The live Excel path still exists and is what warms these stores - see
`citivelo_excel.ipynb` for that, and `scripts/citivelo_excel_intraday_warm.py` for
the warm itself.

## What is warmed

| asset | contents |
|---|---|
| `<curve>-CITIVELOEXCEL` | one EOD curve per day |
| `<curve>-CITIVELOEXCELMIN` | one curve per published **minute** |
| `USD-SWAPTIONVOL-CITIVELOEXCEL` | the swaption cube, ATM + strike offsets |

Curves cover **USD / EUR / GBP / CAD / JPY** over roughly two years; the swaption
cube goes back to **2015-10-08**, ATM-only until Citi began publishing strike
offsets on **2020-01-24**.

Swaps and swaptions are both driven through `TimeseriesBuilder`: an MDP builds the
dated market data, a TB prices and caches a list of queries against it, and the
query says what you want in trade terms. Section 4 is section 2 with a different
product.

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use("ggplot")
pylab.rcParams.update({
    "legend.fontsize": "medium", "figure.figsize": (18, 6),
    "axes.labelsize": "medium", "axes.titlesize": "medium",
    "xtick.labelsize": "medium", "ytick.labelsize": "medium",
})

import datetime
import warnings

import numpy as np
import pandas as pd
import pytz

NYC_tz = pytz.timezone("America/New_York")
warnings.filterwarnings("ignore", category=UserWarning)

import sys
sys.path.append("../../")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from Caching.curve_store import CurveStore
from Caching.swaption_cube_store import SwaptionCubeStore
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedStructure, UnifiedValue
from TB.IRSwapsTB import IRSwapsTB
from TB.IRSwaptionsTB import IRSwaptionsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

CURVES = ["USD-SOFR-1D", "EUR-ESTR-1D", "GBP-SONIA-1D", "CAD-CORRA-1D", "JPY-TONAR-1D-LCH"]

# One MDP, one router, reused by every cell. IRSwapsTB caches the MDP's fetcher
# state, and the fixings a curve needs are resolved once per process rather than
# once per curve - which used to be 97% of a warmed read.
curve_mdp = IRSwapsMDP(source="citivelo_excel_rl")      # ..._ql for a QuantLib curve
irs_tb = IRSwapsTB(curve_mdp, show_tqdm=False, use_ts_cache=False)

# Swaptions ride the SAME TimeseriesBuilder through their own MDP and router, so
# section 4 below is the same three lines as section 2 with a different product.
#
#   source="CITIVELO-RL"   Citi's cube as the vol, rateslib as the pricing
#                          engine. The -QL engine needs a strike smile and so
#                          cannot price the pre-2020 ATM-only days at all.
#   curve_source=...       the DISCOUNT curve, a separate choice from the vol.
#                          It does NOT move normal vol - measured identical to
#                          the last digit against ERIS_EOD_LIVE-RL_BASIC - only
#                          the premium and the greeks, by ~0.06-0.12%.
#   request_defaults       {"verify": False} turns off the cube's node-ordering
#                          assertion, which re-prices all 1,989 nodes: ~236 s for
#                          the first valuation of a day with it, ~1.2 s per value
#                          without. It checks the DATA, not the pricing, and the
#                          tie-out cell below is the check that actually matters
#                          here. It is reachable only through request_defaults -
#                          IRSwaptionsTB builds its bulk request from a fixed key
#                          set and forwards nothing else.
vol_mdp = IRSwaptionMDP(
    source="CITIVELO-RL",
    curve_source="citivelo_excel_rl",
    request_defaults={"verify": False},
)
swaption_tb = IRSwaptionsTB(vol_mdp, show_tqdm=False)

ts = TimeseriesBuilder()

## 1. What is actually warmed

Check coverage before asking for a window - a request outside it silently falls
through to the live Excel path, which is slow and needs a signed-in add-in.

In [3]:
store = CurveStore.default()

rows = []
for curve in CURVES:
    for label, suffix in (("EOD", "CITIVELOEXCEL"), ("1-min", "CITIVELOEXCELMIN")):
        asset = f"{curve}-{suffix}"
        try:
            days = sorted(store.available_dates(asset))
        except Exception:
            days = []
        rows.append({
            "curve": curve, "grid": label, "days": len(days),
            "first": days[0] if days else None, "last": days[-1] if days else None,
        })
coverage = pd.DataFrame(rows)
coverage

,curve,grid,days,first,last
0,USD-SOFR-1D,EOD,5506,2005-01-03,2026-08-07
1,USD-SOFR-1D,1-min,1537,2021-09-14,2026-08-12
2,EUR-ESTR-1D,EOD,5565,2005-01-04,2026-08-07
3,EUR-ESTR-1D,1-min,530,2024-08-01,2026-08-12
4,GBP-SONIA-1D,EOD,4067,2010-11-26,2026-08-07
5,GBP-SONIA-1D,1-min,530,2024-08-01,2026-08-12
6,CAD-CORRA-1D,EOD,2820,2012-01-03,2026-08-07
7,CAD-CORRA-1D,1-min,635,2024-08-01,2026-08-12
8,JPY-TONAR-1D-LCH,EOD,1108,2017-11-09,2026-08-07
9,JPY-TONAR-1D-LCH,1-min,530,2024-08-01,2026-08-12


## 2. EOD, multiple currencies

The ordinary `TimeseriesBuilder` path. A **bare `datetime.date`** is what selects
the end-of-day curve.

> `pandas.Timestamp` subclasses `datetime` subclasses `date`, so
> `pd.Timestamp("2026-07-01")` is *midnight* and also reads as EOD. Anything with a
> real time-of-day routes to the minute store instead.

In [4]:
queries = [
    UnifiedQuery(curve=c, tenor="10Y", value=UnifiedValue.IRS_RATE, name=f"{c} 10Y")
    for c in ["USD-SOFR-1D", "EUR-ESTR-1D", "GBP-SONIA-1D"]
]

eod = ts.get_timeseries(
    start=datetime.date(2025, 1, 1),
    end=datetime.date(2026, 7, 31),
    queries=queries,
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
eod

,EUR-ESTR-1D 10Y,GBP-SONIA-1D 10Y,USD-SOFR-1D 10Y
Date,,,
2025-01-01,2.21415,4.07115,NaN
2025-01-02,2.22281,4.08025,4.07384
2025-01-03,2.27528,4.09088,4.09298
2025-01-06,2.31393,4.11082,4.13481
2025-01-07,2.34613,4.16296,4.20250
...,...,...,...
2026-07-27,2.92582,4.56676,4.21882
2026-07-28,2.90609,4.53995,4.18693
2026-07-29,2.94204,4.60058,4.20542


In [5]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
for col in eod.columns:
    plot(eod[col], which="left")
legend(valfmt="{:.3f}", show_date=True)

### Curve trades read the same way

`tenor="2y/10y"` is a curve, `"2y/5y/10y"` a fly. The weights and signs are applied
by the query, not by you.

In [6]:
spreads = ts.get_timeseries(
    start=datetime.date(2026, 6, 1),
    end=datetime.date(2026, 7, 31),
    queries=[
        UnifiedQuery(curve=c, tenor="2y/10y", value=UnifiedValue.IRS_RATE, name=f"{c} 2s10s")
        for c in ["USD-SOFR-1D", "EUR-ESTR-1D", "GBP-SONIA-1D"]
    ],
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
# IRS_RATE is a DECIMAL for a 1-leg query and already in BP for 2-3 legs, so
# this needs no scaling. Multiplying by 100 here reported EUR 2s10s as
# 2,539 bp instead of 25.4 - the exact trap section 5 warns about.
spreads.tail()

,EUR-ESTR-1D 2s10s,GBP-SONIA-1D 2s10s,USD-SOFR-1D 2s10s
Date,,,
2026-07-27,25.398000,28.322,5.076
2026-07-28,26.866000,30.926,6.367
2026-07-29,24.547999,27.010,12.216
2026-07-30,28.975000,34.364,15.874
2026-07-31,28.217000,34.308,17.718


## 3. Intraday, multiple currencies

Pass a **timezone-aware** datetime and the request routes to the minute store.

Two things worth knowing:

* the add-in stamps **every** curve in New York wall-clock whatever the currency,
  but partitions are keyed by the curve's **local** trading date - the conversion
  is done for you, so just pass an aware timestamp in any zone;
* `meta_data["snapshot_lag_seconds"]` reports how far the served minute sits from
  the one you asked for. Check it rather than assuming an exact hit.

In [7]:
start = NYC_tz.localize(datetime.datetime(2026, 7, 29, 4, 0))
end   = NYC_tz.localize(datetime.datetime(2026, 7, 29, 17, 0))

queries = [ 
		# 	UnifiedQuery(
        #     curve="USD-SOFR-1D",
        #     tenor="IMM_M27xIMM_U27/IMM_M28xIMM_U28",
        #     value=UnifiedValue.IRS_RATE,
        # ),
	UnifiedQuery(
		curve="USD-SOFR-1D",
		tenor="10y",
		value=UnifiedValue.IRS_CITIVELO_SWAP_SPREAD,
		# value=UnifiedValue.IRS_RATE,
	)
]

intraday = ts.get_timeseries(
    start=start, end=end,
    queries=queries,
    freq="1min",
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
print(intraday.shape)
intraday.head()

(781, 1)


,USD-SOFR-1D 10y OUTRIGHT CITIVELO_SWAP_SPREAD
Date,
2026-07-29 04:00:00-04:00,-41.6331
2026-07-29 04:01:00-04:00,-41.6608
2026-07-29 04:02:00-04:00,-41.6287
2026-07-29 04:03:00-04:00,-41.4737
2026-07-29 04:04:00-04:00,-41.4640


In [8]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(intraday["USD-SOFR-1D 10y OUTRIGHT CITIVELO_SWAP_SPREAD"], which="left")
# plot(intraday["EUR-ESTR-1D 10Y"], which="right")
# plot(intraday["GBP-SONIA-1D 10Y"], which="right")
legend(valfmt="{:.4f}", show_date=True)

### One curve, one day, every published minute

The grid is ~12 hours of 1-minute bars per curve per day, in the curve's own local
session.

In [9]:
day = datetime.date(2026, 7, 22)
minutes = ts.get_timeseries(
    start=NYC_tz.localize(datetime.datetime.combine(day, datetime.time(3, 0))),
    end=NYC_tz.localize(datetime.datetime.combine(day, datetime.time(17, 0))),
    queries=[UnifiedQuery(curve="USD-SOFR-1D", tenor="5Y",
                          value=UnifiedValue.IRS_RATE)],
    freq="1min",
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
print(f"{len(minutes)} minute observations")
display(minutes.describe())
display(minutes)

plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(minutes["USD-SOFR-1D 5Y OUTRIGHT RATE"], which="left")
legend(valfmt="{:.4f}", show_date=True)

841 minute observations


,USD-SOFR-1D 5Y OUTRIGHT RATE
count,841.000000
mean,4.102422
std,0.015940
min,4.074820
25%,4.086540
50%,4.103210
75%,4.117150
max,4.126280


,USD-SOFR-1D 5Y OUTRIGHT RATE
Date,
2026-07-22 03:00:00-04:00,4.089650
2026-07-22 03:01:00-04:00,4.089379
2026-07-22 03:02:00-04:00,4.089489
2026-07-22 03:03:00-04:00,4.089241
2026-07-22 03:04:00-04:00,4.089320
...,...
2026-07-22 16:56:00-04:00,4.117310
2026-07-22 16:57:00-04:00,4.117948
2026-07-22 16:58:00-04:00,4.117661


## 4. The swaption cube, through the same MDP/TB/Query path

`SwaptionCubeStore` holds the raw vol grid per day - **ATM and every strike
offset** - and reading it directly gives you the grid and nothing else.

The cells below go through `TimeseriesBuilder` instead, exactly as sections 2 and
3 do for swaps: `IRSwaptionMDP` builds one dated market context per day (Citi's
cube + a discount curve + a pricing engine), `IRSwaptionsTB` prices a list of
queries against it and caches every `(date, query)` cell, and the query says what
you want in trade terms rather than in grid coordinates.

What that buys over `read_day`:

* **strikes, not columns.** `strike="ATMF+50"` is resolved on the day's own
  forward. The grid's `offset_bp` columns are the special case where the two
  coincide - which is exactly what the tie-out cell proves.
* **structures.** A straddle, a payer spread, a vol calendar - one query, one
  number, the risk weights applied by the structure.
* **everything that is not vol.** Premium, DV01, vega, theta, daily breakeven.
  None of it is in the store; all of it needs the curve and the engine that
  `IRSwaptionMDP` already assembled.
* **the same cache discipline as the swap side.** Priced cells are written to
  `IRSwaptionsTB`'s disk cache, so the second run of any of this is a lookup.

It is not free: **~1.2 s per (date, value)** on a cold cache. The windows below
are deliberately short. For long history warm it out of process first -

```
conda run -n stir python scripts/citivelo_swaption_ts_warm.py warm \
    --curve-source citivelo_excel_rl --start 2024-01-01 --chunk-days 21
```

`--curve-source` matters: the value cache stem contains it, and the script
defaults to ERIS, which would warm a stem this notebook never reads.

In [10]:
cube_store = SwaptionCubeStore.default()
VOL_ASSET = "USD-SWAPTIONVOL-CITIVELOEXCEL"
VOL_CURVE = "USD-SOFR-1D"

cube_dates = sorted(cube_store.available_dates(VOL_ASSET))
curve_dates = sorted(store.available_dates(f"{VOL_CURVE}-CITIVELOEXCEL"))

# A swaption VALUE needs a cube AND a discount curve for the same day, and the
# two warms run at different times - the cube is usually a day or two ahead. Ask
# past the curve warm and the request falls through the CurveStore to the
# cached-then-live quotes layer, which is the one path here that can reach Excel.
# So the window ends where BOTH are warm.
VOL_END = min(cube_dates[-1], curve_dates[-1])
VOL_START = datetime.date(2026, 5, 1)

# ...and then ask only for the days the store actually HAS.
#
# A business-day range is not a set of published days: three US market holidays
# fall inside the window below and none of them has a cube. That is not a quiet
# skip - a date the store cannot serve makes the provider warn and then CONNECT
# TO EXCEL to go and fetch it, which is the one thing this notebook promises not
# to do. Measured on the first run of this notebook: 2026-05-25, 2026-06-19 and
# 2026-07-03 each drove a COM fetch and then failed on an ATM-only cube.
curve_day_set = set(curve_dates)
VOL_DAYS = [d for d in cube_dates if VOL_START <= d <= VOL_END and d in curve_day_set]
VOL_TS = [datetime.datetime.combine(d, datetime.time()) for d in VOL_DAYS]

skipped = sorted(set(pd.bdate_range(VOL_START, VOL_END).date) - set(VOL_DAYS))

print(f"cube  {len(cube_dates)} days  {cube_dates[0]} .. {cube_dates[-1]}")
print(f"curve {len(curve_dates)} days  {curve_dates[0]} .. {curve_dates[-1]}")
print(f"priceable window: {VOL_START} .. {VOL_END}  ->  {len(VOL_DAYS)} days")
print(f"business days with no cube ({len(skipped)}): {[d.isoformat() for d in skipped]}")

one_day = cube_store.read_day(VOL_ASSET, VOL_END)
print(f"\n{one_day.shape[0]} rows for {VOL_END}")
print("expiries:", list(one_day["expiry"].unique()))
print("tenors  :", list(one_day["tenor"].unique()))
print("offsets :", sorted(one_day["offset_bp"].unique()))
one_day.head()

cube  2701 days  2015-10-08 .. 2026-08-11
curve 5506 days  2005-01-03 .. 2026-08-07
priceable window: 2026-05-01 .. 2026-08-07  ->  68 days
business days with no cube (3): ['2026-05-25', '2026-06-19', '2026-07-03']

1989 rows for 2026-08-07
expiries: ['1M', '2M', '3M', '6M', '9M', '1Y', '18M', '2Y', '3Y', '4Y', '5Y', '7Y', '10Y', '12Y', '15Y', '20Y', '30Y']
tenors  : ['1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '15Y', '20Y', '30Y']
offsets : [np.float64(-200.0), np.float64(-100.0), np.float64(-75.0), np.float64(-50.0), np.float64(-25.0), np.float64(-10.0), np.float64(0.0), np.float64(10.0), np.float64(25.0), np.float64(50.0), np.float64(75.0), np.float64(100.0), np.float64(200.0)]


,expiry,tenor,offset_bp,vol_bp,as_of,currency,measure,skew_measure,served_unit,vol_unit,strike_unit,source,citi_index,schema_version,asset,date
0,1M,1Y,-200.0,169.0770,2026-08-07,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-07
1,1M,1Y,-100.0,116.1420,2026-08-07,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-07
2,1M,1Y,-75.0,102.3490,2026-08-07,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-07
3,1M,1Y,-50.0,88.9284,2026-08-07,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-07
4,1M,1Y,-25.0,77.7327,2026-08-07,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-07


### ATM normal vol

`shorthand="3Mx10Y"` is the expiry and the tail; `strike="ATMF"` resolves to the
day's own forward. `NVOL` comes back in **normal basis-point vol**, the same
units as the store's `vol_bp`.

A straddle is used rather than a single leg because ATM vol is a property of the
point, not of a side - `NVOL` on a two-leg package is the weight-averaged leg
vol, and both legs sit on the same strike.

In [11]:
PAIRS = ["3Mx10Y", "1Yx10Y", "5Yx10Y", "3Mx2Y"]

atm_queries = [
    UnifiedQuery(
        curve=VOL_CURVE,
        selector={"shorthand": sh, "strike": "ATMF"},
        structure=UnifiedStructure.IRSWAPTION_STRADDLE,
        value=UnifiedValue.IRSWAPTION_NVOL,
        name=f"{sh} ATM nvol",
    )
    for sh in PAIRS
]

atm = ts.get_timeseries(
    start=VOL_START,
    end=VOL_END,
    timestamps=VOL_TS,
    queries=atm_queries,
    routers={"IRSWAPTION": swaption_tb},
    n_jobs=4,
)
print(atm.shape)
atm.tail()

(68, 4)


,1Yx10Y ATM nvol,3Mx10Y ATM nvol,3Mx2Y ATM nvol,5Yx10Y ATM nvol
Date,,,,
2026-08-03,82.8158,76.9877,94.8586,85.6716
2026-08-04,81.5841,75.1800,92.0633,84.7934
2026-08-05,80.9518,74.2386,90.9156,84.5367
2026-08-06,82.1221,77.2107,96.3277,84.8101
2026-08-07,80.4086,73.6210,92.6413,84.2432


In [12]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
for col in atm.columns:
    plot(atm[col], which="left")
legend(valfmt="{:.1f}", show_date=True)

### Skew, as strikes rather than as grid columns

`strike="ATMF+50"` means *50 bp above this day's forward*, resolved per day. The
store's `offset_bp=+50` column means the same thing, which is why the two agree -
but the query form is the one that keeps working for `ATMF+37`, for a delta
strike, or for an expiry the grid does not publish.

A signed strike also picks the side: `ATMF+50` is a payer, `ATMF-50` a receiver.
That inference is in `IRSwaptionQuery`, so the `structure=` below is only being
explicit about what it would have chosen anyway.

In [13]:
OFFSETS = [-100, -50, 50, 100]

skew_queries = [
    UnifiedQuery(
        curve=VOL_CURVE,
        selector={"shorthand": "3Mx10Y", "strike": "ATMF"},
        structure=UnifiedStructure.IRSWAPTION_STRADDLE,
        value=UnifiedValue.IRSWAPTION_NVOL,
        name="+0bp",
    )
] + [
    UnifiedQuery(
        curve=VOL_CURVE,
        selector={"shorthand": "3Mx10Y", "strike": f"ATMF{off:+d}"},
        structure=(UnifiedStructure.IRSWAPTION_PAYER if off > 0
                   else UnifiedStructure.IRSWAPTION_RECEIVER),
        value=UnifiedValue.IRSWAPTION_NVOL,
        name=f"{off:+d}bp",
    )
    for off in OFFSETS
]

skew = ts.get_timeseries(
    start=VOL_START,
    end=VOL_END,
    timestamps=VOL_TS,
    queries=skew_queries,
    routers={"IRSWAPTION": swaption_tb},
    n_jobs=4,
)
# Quote the wings against ATM: the level moves far more than the shape.
for col in [c for c in skew.columns if c != "+0bp"]:
    skew[f"{col} - ATM"] = skew[col] - skew["+0bp"]
skew[[c for c in skew.columns if c.endswith("- ATM")]].tail()

,+100bp - ATM,+50bp - ATM,-100bp - ATM,-50bp - ATM
Date,,,,
2026-08-03,31.3763,12.7720,18.2988,3.7962
2026-08-04,28.4970,11.5172,16.0303,3.0728
2026-08-05,28.5884,11.5740,16.1926,3.1537
2026-08-06,28.3343,11.4193,15.6564,2.8822
2026-08-07,27.6060,11.1514,15.3981,2.8988


In [14]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
for col in [c for c in skew.columns if c.endswith("- ATM")]:
    plot(skew[col], which="left")
plot(skew["+0bp"], which="right")
legend(valfmt="{:.2f}", show_date=True)

### Tie-out: the query path against the raw grid

The one check worth running before trusting any of the above. `ATMF+50` is
resolved on the discount curve's forward and then looked up in the cube; the grid
cell is Citi's own `+50` quote. If those two forwards disagreed, every wing would
be read off the wrong part of the smile - and the error would be invisible,
because a plausible number comes back either way.

Measured over the window below: identical to **1e-14**, at ATM and at all four
wings. That also means `NVOL` here is a pure grid read - it does not depend on
the discount curve, which is why `curve_source` moves the premium and the greeks
but not the vol.

In [15]:
# The index holds whatever the row was PRICED with: a datetime for a freshly
# computed cell (the request is driven off VOL_TS) and a plain date for one served
# from the TB cache, which stores the reference point it was built with. Both are
# the same instant - _dt_to_epoch_ns normalises them, which is why the cache hits
# across the two - but SwaptionCubeStore partitions on a DATE, and
# Timestamp.isoformat() would build a "date=2026-08-07T00:00:00" path that matches
# nothing. Normalise before reading.
rows = []
for d in list(skew.index[-3:]):
    day = d.date() if isinstance(d, datetime.datetime) else d
    frame = cube_store.read_day(VOL_ASSET, day)
    sl = frame[(frame["expiry"] == "3M") & (frame["tenor"] == "10Y")]
    for off in [0] + OFFSETS:
        cell = sl[sl["offset_bp"] == off]
        rows.append({
            "Date": d,
            "offset": f"{off:+d}bp",
            "query": float(skew.loc[d, f"{off:+d}bp"]),
            "grid": float(cell["vol_bp"].iloc[0]) if len(cell) else np.nan,
        })

tie = pd.DataFrame(rows)
tie["diff"] = tie["query"] - tie["grid"]
print(f"max |diff| = {tie['diff'].abs().max():.2e} bp")
tie

max |diff| = 2.84e-14 bp


,Date,offset,query,grid,diff
0,2026-08-05,+0bp,74.2386,74.2386,0.000000e+00
1,2026-08-05,-100bp,90.4312,90.4312,0.000000e+00
2,2026-08-05,-50bp,77.3923,77.3923,0.000000e+00
3,2026-08-05,+50bp,85.8126,85.8126,-1.421085e-14
4,2026-08-05,+100bp,102.8270,102.8270,0.000000e+00
5,2026-08-06,+0bp,77.2107,77.2107,-1.421085e-14
6,2026-08-06,-100bp,92.8671,92.8671,1.421085e-14
7,2026-08-06,-50bp,80.0929,80.0929,-1.421085e-14
8,2026-08-06,+50bp,88.6300,88.6300,0.000000e+00
9,2026-08-06,+100bp,105.5450,105.5450,-2.842171e-14


### What the grid cannot give you

Premium and greeks are not in the cube. They need the discount curve and the
engine, which is what `IRSwaptionMDP` assembled alongside the vol - so they are
the same query with a different `value`.

**Units, because none of these are the same and all of them look like numbers:**

| value | unit |
|---|---|
| `NVOL` | normal vol, bp/yr |
| `SPOT_PREM` | bp of notional |
| `VEGA_01`, `THETA_1D` | currency, on the provider's 1e8 notional |
| `DAILY_BREAKEVEN_NVOL` | bp/day - the daily move that pays for one day of theta |

In [16]:
book = ts.get_timeseries(
    start=VOL_START,
    end=VOL_END,
    timestamps=VOL_TS,
    queries=[
        UnifiedQuery(curve=VOL_CURVE, selector={"shorthand": "3Mx10Y", "strike": "ATMF"},
                     structure=UnifiedStructure.IRSWAPTION_STRADDLE,
                     value=UnifiedValue.IRSWAPTION_SPOT_PREM, name="3Mx10Y prem (bp)"),
        UnifiedQuery(curve=VOL_CURVE, selector={"shorthand": "3Mx10Y", "strike": "ATMF"},
                     structure=UnifiedStructure.IRSWAPTION_STRADDLE,
                     value=UnifiedValue.IRSWAPTION_VEGA_01, name="3Mx10Y vega01 ($)"),
        UnifiedQuery(curve=VOL_CURVE, selector={"shorthand": "3Mx10Y", "strike": "ATMF"},
                     structure=UnifiedStructure.IRSWAPTION_STRADDLE,
                     value=UnifiedValue.IRSWAPTION_THETA_1D, name="3Mx10Y theta1d ($)"),
        UnifiedQuery(curve=VOL_CURVE, selector={"shorthand": "3Mx10Y", "strike": "ATMF"},
                     structure=UnifiedStructure.IRSWAPTION_STRADDLE,
                     value=UnifiedValue.IRSWAPTION_DAILY_BREAKEVEN_NVOL,
                     name="3Mx10Y breakeven (bp/day)"),
    ],
    routers={"IRSWAPTION": swaption_tb},
    n_jobs=4,
)
book.tail()

,3Mx10Y breakeven (bp/day),3Mx10Y prem (bp),3Mx10Y theta1d ($),3Mx10Y vega01 ($)
Date,,,,
2026-08-03,0.839109,247.862949,-13507.618245,32195.136299
2026-08-04,0.819407,242.820361,-13232.815741,32298.531666
2026-08-05,0.809146,239.924470,-13075.000319,32318.021843
2026-08-06,0.841540,248.815060,-13559.504766,32225.463575
2026-08-07,0.785296,240.166839,-12808.989392,32622.056023


### A vol calendar, priced from cells you already have

`1Yx10Y - 3Mx10Y` in normal vol. Both legs were priced by the ATM cell above, so
this costs nothing to add: the TB cache is keyed on the query, not on the frame
it was asked for.

In [17]:
cal = ts.get_timeseries(
    start=VOL_START,
    end=VOL_END,
    timestamps=VOL_TS,
    queries=[
        UnifiedQuery(curve=VOL_CURVE, selector={"shorthand": "1Yx10Y", "strike": "ATMF"},
                     structure=UnifiedStructure.IRSWAPTION_STRADDLE,
                     value=UnifiedValue.IRSWAPTION_NVOL, name="1Yx10Y ATM nvol"),
        UnifiedQuery(curve=VOL_CURVE, selector={"shorthand": "3Mx10Y", "strike": "ATMF"},
                     structure=UnifiedStructure.IRSWAPTION_STRADDLE,
                     value=UnifiedValue.IRSWAPTION_NVOL, name="3Mx10Y ATM nvol"),
    ],
    routers={"IRSWAPTION": swaption_tb},
    n_jobs=4,
)
cal["1Y-3M x 10Y"] = cal["1Yx10Y ATM nvol"] - cal["3Mx10Y ATM nvol"]

plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(cal["1Y-3M x 10Y"], which="left")
plot(cal["3Mx10Y ATM nvol"], which="right")
legend(valfmt="{:.2f}", show_date=True)

### Where the numbers came from

`IRSwaptionMDP` attaches the cube's provenance to the context metadata, so
"warmed store or live Excel?" and "full smile or ATM-only day?" are readable
rather than inferred. `origin='swaption_cube_store'` is the whole claim of this
notebook, in one field.

In [18]:
ctx = vol_mdp.get_data({"curve_name": VOL_CURVE, "timestamp": VOL_END})
print(ctx.id())
ctx.meta()["citivelo_provenance"]

USD-SOFR-1D|2026-08-07|CITIVELO-RL|atmf_normal


{'origin': 'swaption_cube_store',
 'asset': 'USD-SWAPTIONVOL-CITIVELOEXCEL',
 'as_of': '2026-08-07',
 'smile': 'full',
 'n_offsets': 13,
 'offsets_bp': [-200.0,
  -100.0,
  -75.0,
  -50.0,
  -25.0,
  -10.0,
  0.0,
  10.0,
  25.0,
  50.0,
  75.0,
  100.0,
  200.0],
 'store_source': 'citivelo_excel_warm/DAILY'}

### The smile and the ATM surface on one date

Kept as a direct store read. This is a display of the whole published grid - 153
ATM cells and 1,989 rows in all - and pushing that through the pricer would cost
minutes to reproduce numbers the tie-out cell has already shown are the same.
Query the points you want a *timeseries* of; read the store when you want to look
at the grid.

In [19]:
snap = cube_store.read_day(VOL_ASSET, cube_dates[-1])
smile = (snap[(snap["expiry"] == "3M") & (snap["tenor"] == "10Y")]
         .sort_values("offset_bp")[["offset_bp", "vol_bp"]]
         .set_index("offset_bp"))

surface = (snap[snap["offset_bp"] == 0]
           .pivot_table(index="expiry", columns="tenor", values="vol_bp"))

print(f"3Mx10Y smile on {cube_dates[-1]}")
display(smile.T)
print("\nATM surface (normal bp vol)")
display(surface)

3Mx10Y smile on 2026-08-11


offset_bp,-200.0,-100.0,-75.0,-50.0,-25.0,-10.0,0.0,10.0,25.0,50.0,75.0,100.0,200.0
vol_bp,119.791,90.0053,83.2063,77.5625,74.2769,74.0518,74.7383,76.0616,79.0524,85.8588,93.8504,102.313,136.449



ATM surface (normal bp vol)


tenor,10Y,15Y,1Y,20Y,2Y,30Y,3Y,5Y,7Y
expiry,,,,,,,,,
10Y,83.3464,80.7371,87.1961,79.7242,86.8876,78.3584,86.4739,85.6346,84.7501
12Y,82.1382,79.6067,86.4299,78.4141,86.2145,77.3302,85.6174,84.4132,83.5267
15Y,80.2891,77.8407,85.4885,76.3575,85.3395,75.6886,84.4435,82.6118,81.6870
18M,81.8289,79.3482,97.8722,77.1929,94.5644,75.7613,92.4212,87.8017,84.9126
1M,73.1226,69.3130,82.7097,66.5593,92.4348,64.3021,91.1554,86.6534,80.9725
1Y,80.9729,78.1403,96.1533,75.6126,95.2298,73.9978,93.0362,87.6834,84.5081
20Y,77.7208,75.3765,83.5442,74.0705,83.3096,73.4443,82.2123,79.9785,79.0863
2M,73.3814,69.5151,85.7325,66.8765,94.0180,64.5132,92.0875,84.4504,79.7470
2Y,83.0614,80.8778,96.5055,78.9923,93.8948,77.6710,91.4399,88.0843,85.5669


## 5. Gotchas

Each of these produced a confident wrong number during development.

**Units.** `curve.fair_rate()` returns a **decimal**; Citi quotes **percent**.
A draft of this very notebook scaled the 2s10s cell by 100 and printed EUR 2s10s as
**2,539 bp** instead of 25.4.
Comparing them unscaled reads as a 442 bp error on a curve that is exact. Through
`UnifiedValue.IRS_RATE` a 1-leg query is decimal and a 2-3 leg query is **bp** -
which is why the curve cell above multiplies by 100 and the outright does not.

**Spot lag.** If you build instruments yourself, take the lag from
`SettlementDays`, not `payment_lag`. They coincide for USD (both 2) and diverge for
EUR (1 vs 2); the wrong one reported 0.4358 bp at EUR 3M where the right one gives
0.0006 bp.

**`bulk_get_data` is not a fast path here.** It has branches for CME / ERIS / SDR /
BARCHART but none for CITIVELO, so it falls through to a generic per-point loop -
and for EOD it is *slower* than looping `get_data`. `TimeseriesBuilder` as used
above is the right entry point.

**`ignore_cache=True` reaches Excel.** It bypasses the store and rebuilds through
the cached-then-live quotes layer. Leave it off unless you mean it.

**Coverage is not uniform.** Check section 1 before choosing a window; a date
outside the warm silently takes the live path.

### Swaptions

**`verify=True` is 236 s/day.** The default on the Citi cube asserts the node
ordering by re-pricing all 1,989 nodes. Profiled on one USD date: the first
valuation took 235.8 s, of which 236.8 s *was* the assertion, and the next took
1.54 s. `request_defaults={"verify": False}` is the only route to the flag -
`IRSwaptionsTB` builds its `bulk_get_data` request from a fixed key set and
forwards nothing else. It is a check on the data, not on the pricing.

**A business-day range is not a set of published days.** `pd.bdate_range` is what
`start=`/`end=` expands to, and it contains US market holidays. The cube has no
partition for those, and a date the store cannot serve does not skip quietly - the
provider warns and then **connects to Excel over COM** to fetch it. The first run
of this notebook did exactly that on 2026-05-25, 06-19 and 07-03, drove three COM
fetches, and then failed each one on an ATM-only cube. Pass `timestamps=` built
from `cube_store.available_dates`, as the coverage cell does.

**Two warms, two end dates.** The cube warm and the curve warm run separately and
the cube is usually ahead - 2026-08-11 against 2026-08-07 as this was written. A
day with a cube and no curve has the same failure mode as a holiday. Clamp the
window to `min(cube_dates[-1], curve_dates[-1])`, as the coverage cell does.

**`curve_source` does not move normal vol.** Measured against
`ERIS_EOD_LIVE-RL_BASIC` over 2026-08-04..08-07: `NVOL` identical to the last
digit at ATM and at +50, `SPOT_PREM` and `VEGA_01` different by 0.06-0.12%. The
vol is a read of Citi's grid; the discount curve only prices the premium. Do not
conclude from a matching vol column that the two configurations agree.

**The value cache stem contains `curve_source`.** Two `curve_source` settings
cache separately and neither invalidates the other, so switching one reprices
everything - and `scripts/citivelo_swaption_ts_warm.py` defaults to ERIS, which
warms a stem this notebook does not read. Pass
`--curve-source citivelo_excel_rl`.

**One `UserWarning` about fixings is the degrade working.** With Excel closed the
published-fixings lookup asks once per process, fails, warns, and re-serves the
cached series. The staleness gate still refuses a series that does not reach the
date being priced, so this cannot silently under-fix a float leg.

**Pre-2020 is ATM-only.** Citi published no strike offsets before 2020-01-24, so
1,067 of the 2,701 stored USD days have a single offset. `source="CITIVELO-QL"`
cannot build a cube from those at all; `-RL` can. A day like that looks exactly
like a cache miss - check `citivelo_provenance['smile']` before refetching.